# Forberegn segmenteringsmasker (Colab)

Gjør den tunge ML-jobben i Colab. Gjennomgang gjøres etterpå på din PC.

**Kjør cellene i rekkefølge:**

1. **Oppsett** — mount Drive, klon kode fra GitHub, installer avhengigheter
2. **Konfigurasjon** — stier og hjelpefunksjoner
3. **Last inn modeller** — SAM2 og SegFormer (kjøres én gang per økt)
4. **Forberegn masker** — tung ML; trygt å avbryte og fortsette
5. **Last ned cache** — pakker og laster ned zip-fil til PC

**På lokal PC etterpå:**
```
# Pakk ut zip-filen i prosjektmappen (slik at .precompute/ havner i data/annotated_seg/)
Expand-Archive -Path precompute_cache.zip -DestinationPath data\annotated_seg\

# Kjør gjennomgang (ingen ML — nær-instant mellom bilder)
py -3.14 -m src.labeling.seg_cache_review
```

In [ ]:
import os, subprocess, sys
from pathlib import Path

# --- 1. Mount Google Drive (dataen din er her) ---
from google.colab import drive
drive.mount('/content/drive')

# Sti til prosjektmappen på Drive (der data/ ligger) — juster om nødvendig
DRIVE_PROJECT = Path("/content/drive/MyDrive/trash-bin-detection")

# --- 2. Klon kode fra GitHub ---
REPO_URL = "https://github.com/davgei/trash-bin-detection.git"
REPO_DIR = Path("/content/trash-bin-detection")
BRANCH   = "streetview-yolo-retry"

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True
    )
    print(f"Klonet {BRANCH} til {REPO_DIR}")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    print(f"Oppdaterte eksisterende klon i {REPO_DIR}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# --- 3. Installer avhengigheter ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "ultralytics", "transformers"], check=True)

print(f"Kode: {REPO_DIR}")
print(f"Data: {DRIVE_PROJECT}")
print("Oppsett OK.")

## Steg 2 — Konfigurasjon

In [ ]:
import csv
import pickle
from pathlib import Path

import cv2

from src.labeling.sam2_seg_autolabel import (
    BIN_CLIP_MARGIN,
    DEFAULT_SAM_MODEL,
    DEFAULT_SEMANTIC_MODEL,
    compute_seg_for_image,
    list_split_images,
    load_sam,
    load_semantic,
    make_preview,
    prepare_output_dirs,
    read_bin_boxes,
)

SOURCE    = DRIVE_PROJECT / "data" / "annotated"      # bilder + boks-labels (kilde)
OUTPUT    = DRIVE_PROJECT / "data" / "annotated_seg"  # segmenteringsdatasett (mål)
CACHE_DIR = OUTPUT / ".precompute"                    # pkl + preview-cache
SPLITS    = ("train", "val", "test")

SAM_MODEL      = DEFAULT_SAM_MODEL
SEMANTIC_MODEL = DEFAULT_SEMANTIC_MODEL
CLIP_MARGIN    = BIN_CLIP_MARGIN


def _label_path(split: str, stem: str) -> Path:
    return OUTPUT / "labels" / split / f"{stem}.txt"


def _skip_path(split: str, stem: str) -> Path:
    return OUTPUT / "labels" / split / f"{stem}.skip"


def _cache_pkl(split: str, stem: str) -> Path:
    d = CACHE_DIR / split
    d.mkdir(parents=True, exist_ok=True)
    return d / f"{stem}.pkl"


def _cache_preview(split: str, stem: str) -> Path:
    return CACHE_DIR / split / f"{stem}.jpg"


if not SOURCE.exists():
    print(f"ADVARSEL: kildemappe ikke funnet: {SOURCE}")
    print("Sjekk at DRIVE_PROJECT peker på riktig mappe (Steg 1).")
else:
    print(f"Kilde:  {SOURCE}")
    print(f"Output: {OUTPUT}")
    print(f"Cache:  {CACHE_DIR}")
    print("Konfigurasjon OK.")

## Steg 3 — Last inn modeller

Tar noen sekunder. Kjøres én gang per økt.

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Enhet: {device}")

print(f"Laster SAM2 ({SAM_MODEL}) ...")
sam = load_sam(SAM_MODEL)

print(f"Laster semantisk modell ({SEMANTIC_MODEL}) ...")
processor, semantic, ground_ids = load_semantic(SEMANTIC_MODEL, device)

if not ground_ids:
    print("ADVARSEL: ingen ground-klasser funnet.")
else:
    names = ", ".join(semantic.config.id2label[i] for i in ground_ids)
    print(f"Ground-klasser: {names}")

print("Modeller lastet.")

## Steg 4 — Forberegn masker

Kjører SAM2 + SegFormer på alle bilder som ikke er godkjent, hoppet over eller cachet.  
Avbryt når som helst med **Kernel › Interrupt** — cachede bilder hoppes over neste gang.

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
prepare_output_dirs(OUTPUT, SPLITS)

pending = []
n_done = n_cached = 0
for split in SPLITS:
    for image_path in list_split_images(SOURCE, split):
        stem = image_path.stem
        if _label_path(split, stem).exists() or _skip_path(split, stem).exists():
            n_done += 1
            continue
        if _cache_pkl(split, stem).exists() and _cache_preview(split, stem).exists():
            n_cached += 1
            continue
        pending.append((split, image_path))

print(f"Allerede godkjent/hoppet over : {n_done}")
print(f"Allerede i cache              : {n_cached}")
print(f"Skal beregnes nå              : {len(pending)}")

for i, (split, image_path) in enumerate(pending):
    stem = image_path.stem
    print(f"[{i + 1}/{len(pending)}] {split}/{image_path.name}", end=" ... ", flush=True)
    image = cv2.imread(str(image_path))
    if image is None:
        print("KAN IKKE LESES — hopper over")
        continue
    height, width = image.shape[:2]
    src_label = SOURCE / "labels" / split / f"{stem}.txt"
    boxes = read_bin_boxes(src_label, width, height)
    try:
        comp = compute_seg_for_image(
            image, boxes, sam, image_path,
            processor, semantic, ground_ids, device, CLIP_MARGIN,
        )
        entry = {
            "split": split,
            "image_filename": image_path.name,
            "boxes": [list(b) for b in boxes],
            "bin_polygons": comp.bin_polygons,
            "ground_polygons": comp.ground_polygons,
            "flagged": bool(comp.flagged),
            "flags": list(comp.flags),
        }
        with open(_cache_pkl(split, stem), "wb") as fh:
            pickle.dump(entry, fh, protocol=5)
        preview = make_preview(image, comp.clipped_masks, comp.ground_mask, boxes, comp.flagged)
        cv2.imwrite(str(_cache_preview(split, stem)), preview)
        print("OK" + ("  [FLAGGET]" if comp.flagged else ""))
    except Exception as exc:
        print(f"FEIL: {exc}")

n_ready = sum(
    1 for split in SPLITS
    for p in (CACHE_DIR / split).glob("*.pkl")
    if not _label_path(split, p.stem).exists() and not _skip_path(split, p.stem).exists()
)
print(f"\nForberegning ferdig. {n_ready} bilde(r) klare for nedlasting og gjennomgang.")

## Steg 5 — Last ned cache

Pakker `.precompute/`-mappen til en zip-fil og laster den ned til PC-en din.  

Pakk ut på PC med PowerShell (fra prosjektmappen):
```powershell
Expand-Archive -Path $env:USERPROFILE\Downloads\precompute_cache.zip -DestinationPath data\annotated_seg\
```
Deretter:
```
py -3.14 -m src.labeling.seg_cache_review
```

In [ ]:
import shutil
from google.colab import files

ZIP_BASE = "/content/precompute_cache"

print("Pakker cache ...")
# make_archive(base_name, format, root_dir, base_dir)
# Zips CACHE_DIR itself, so the zip contains .precompute/train/... etc.
shutil.make_archive(ZIP_BASE, "zip", str(CACHE_DIR.parent), CACHE_DIR.name)
zip_path = ZIP_BASE + ".zip"

import os
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"Zip: {zip_path}  ({size_mb:.1f} MB)")
print("Laster ned ...")
files.download(zip_path)
print("Ferdig. Pakk ut i data/annotated_seg/ på lokal PC.")